# Forecast Lab

Measure whether a strategy's **ranking** carries information, without simulating
a book.

That distinction is the whole reason this notebook exists. A backtest reports
the *product* of two things — does the signal order the cross-section, and does
that ordering survive being turned into a book — and never their difference.
Round one found a strategy that ranks the cross-section well and has a
**negative** decile spread. One equity curve cannot say that; it just looks bad.

Nothing here trains. Evaluation is about the signal a strategy already emits,
so every cell works on the rule-based strategies too.

## 1. Pin a universe

Same `Lab` as the other notebooks, and the same rule: the universe is pinned
once, so every number below is about the strategies rather than about the draw.

In [ ]:
%load_ext autoreload
%autoreload 2

from portfolio_agent.lab import Lab

lab = Lab(universe_size=60)
print(lab)
print("fingerprint:", lab.fingerprint)

In [ ]:
# Write it down, so this experiment can be repeated exactly on any machine.
lab.save_universe("universe/forecast_lab.json")

## 2. One strategy

`evaluate` returns a `ForecastEvaluation`. `render()` is the human view; `.notes`
is the part worth reading first.

In [ ]:
result = lab.evaluate("momentum", horizon=5)
print(result.render())

### What the run did *not* have

Three inputs change what the number **means**, and a run without them says so
rather than leaving you to remember. This is not a to-do list — each line is a
property of the number printed above.

In [ ]:
for note in result.notes:
    print(f"- {note}\n")

## 3. Is it the strategy, or an exposure it carries?

`evaluate` says the ranking worked. It does not say *why*. A low-volatility
screen that ranks well may simply be a beta bet — which is close to
tautological, and worth seeing as a number rather than assuming.

Round one measured momentum at **58% factor loading** and low volatility at
**71%**.

In [ ]:
residual = lab.neutralized("momentum", horizon=5)
print(residual.render())

## 4. How fast does it go stale?

The decay curve decides the rebalance frequency, and therefore the cost. A
signal whose IC is flat to day 21 keeps essentially all of its edge under a
monthly rebalance; one that decays by day 3 does not, and pays the Indian
round trip — **0.79% at 25 bps a side** — to find that out.

In [ ]:
from portfolio_agent.evaluation import decay_curve

decay = decay_curve(
    lab.config, "momentum", universe=lab.tickers,
    horizons=[1, 2, 3, 5, 10, 21],
)
print(decay.render())

## 5. Several strategies, one table

One universe, one set of dates, one table. Comparing runs that each drew their
own sample differs by the draw at least as much as by the strategy — which is
why this is a single call rather than a loop you write yourself.

In [ ]:
table = lab.compare_forecasts(
    ["momentum", "residual_momentum", "low_volatility_idio", "bab", "reversal"],
    horizon=5,
)
table[["strategy", "mean_ic", "icir", "t_stat", "p_value", "spread", "n_dates"]]

### Reading that table

- **`mean_ic`** is the mean *per-date* rank correlation. Not pooled: a pooled
  figure measures whether the score tracks the market's level, and on a signal
  that orders every date perfectly while its level runs the other way, the two
  disagree by nearly 2.0.
- **`t_stat`** is Newey–West corrected for overlapping labels. A 5-day label
  sampled daily overlaps its next four neighbours, and an uncorrected t on
  those is optimistic.
- **`spread`** is net of the Indian cost schedule by default. A gross spread is
  the number that looks best and means least.

## 6. Does the edge survive becoming a book?

The decile spread silently equal-weights. That is a real allocation rule and a
defensible default — but it is a *choice*, and until you compare it with the
alternatives it is an unexamined one.

`mean_variance` is included so you can watch it concentrate: handed a noisy
expected return, an optimizer will put the book in whichever name sampled
luckiest.

In [ ]:
import pandas as pd

from portfolio_agent.evaluation import build_forecast_panel, compare_schemes
from portfolio_agent.strategies.registry import load_strategy
from portfolio_agent.config.schema import StrategyConfig

panel = build_forecast_panel(
    lab.config, load_strategy(StrategyConfig(type="momentum", params={})),
    lab.tickers, horizon=5, keep_prices=True,
)
returns = panel.pivot_table(
    index="date", columns="symbol", values="close", aggfunc="last"
).pct_change()

compare_schemes(panel, returns=returns)[
    ["book_scheme", "book_mean_names_held", "book_mean_max_weight",
     "book_annualized_volatility", "book_sharpe", "book_mean_turnover"]
]

## 7. When did it pay?

A pooled IC made of a strong down-market number and a flat up-market one
describes neither state. This matters most for the low-risk anomaly, where
2025 Asian work finds the effect concentrated in downturns.

Note the conditioner. `realized` splits on the market's return *over the label
horizon* — it says when the signal paid, and is **not tradable**, because on
the decision date nobody knows which bucket the date will land in.

In [ ]:
from portfolio_agent.evaluation import conditional_ic, conditional_notes

split = conditional_ic(panel, horizon=5)
for line in conditional_notes(split):
    print(f"- {line}\n")

## 8. Provenance

Every `evaluate` writes a manifest under `runs/` — the universe, the config, the
git commit, the flags. A metric whose universe and commit are unrecorded is one
nobody can check later, including you.

```bash
portfolio-agent report --run <id>
```

In [ ]:
print("run id:", result.run_id)
print("manifest:", result.manifest_path)

## Where to go next

| Question | Where |
| --- | --- |
| How does this strategy train and backtest? | `01_strategy_lab.ipynb` |
| Which settings are better? | `02_compare_and_sweep.ipynb` |
| What do the missing data files look like? | `docs/OBTAINING_DATA.md` |
| Why is the headline not an equity curve? | `README.md` → Measuring a strategy |